In [14]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from sklearn.metrics import adjusted_rand_score

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

In [15]:
sample = "Xenium_5K_BC"

In [16]:
# paths
intermediate_path = f"../../data/{sample}/intermediate_data/"
processed_path = f"../../data/{sample}/processed_data/"
output_path = f"../../output/1_exploration/{sample}/"

# read data
adata = sc.read_h5ad(intermediate_path + "adata.h5ad")
cell_ids = np.load(processed_path + "cell_ids.npy", allow_pickle=True)
assert adata.shape[0] > cell_ids.shape[0], "cell_ids should be a subset of adata.obs_names"
adata_tumor = adata[adata.obs["cell_id"].isin(cell_ids)].copy()

adata_tumor_nuc = sc.read_h5ad(output_path + "adata_nuclear_embedded.h5ad")
adata_tumor_cyto = sc.read_h5ad(output_path + "adata_cytoplasmic_embedded.h5ad")

assert adata_tumor.shape[0] == adata_tumor_nuc.shape[0], "adata_tumor and adata_tumor_nuc should have the same number of cells"
assert adata_tumor_nuc.shape[0] == adata_tumor_cyto.shape[0], "Nuclear and cytoplasmic embeddings should have the same number of cells"

In [17]:
# Clustering
res_nuc, res_cyto = 0.8, 1.2
sc.tl.louvain(adata_tumor_nuc, resolution = res_nuc, key_added = "subtype_louvain")
sc.tl.louvain(adata_tumor_cyto, resolution = res_cyto, key_added = "subtype_louvain")

adata_tumor.obs["subtype_louvain_nuc"] = adata_tumor_nuc.obs["subtype_louvain"].values
adata_tumor.obs["subtype_louvain_cyto"] = adata_tumor_cyto.obs["subtype_louvain"].values

print(f"Number of clusters (nuclear): {adata_tumor.obs['subtype_louvain_nuc'].nunique()}")
print(f"Number of clusters (cytoplasmic): {adata_tumor.obs['subtype_louvain_cyto'].nunique()}")

Number of clusters (nuclear): 7
Number of clusters (cytoplasmic): 8


In [18]:
# UMAP plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.umap(adata_tumor_nuc, color="subtype_louvain", ax=axes[0], show=False, title="Nuclear expression")
sc.pl.umap(adata_tumor_cyto, color="subtype_louvain", ax=axes[1], show=False, title="Cytoplasmic expression")
plt.savefig(output_path + "umap_clustering.png", dpi=300, bbox_inches="tight")
plt.close()

In [19]:
# ARI
df = pd.DataFrame({"subtype_louvain_nuc": adata_tumor.obs["subtype_louvain_nuc"], "subtype_louvain_cyto": adata_tumor.obs["subtype_louvain_cyto"]})
df.to_csv(output_path + "clustering_results.csv", index=False)

ari = adjusted_rand_score(adata_tumor.obs["subtype_louvain_nuc"], adata_tumor.obs["subtype_louvain_cyto"])
print(f"Adjusted Rand Index (ARI) between nuclear and cytoplasmic clustering: {ari:.4f}")

Adjusted Rand Index (ARI) between nuclear and cytoplasmic clustering: 0.0720


In [20]:
# Scatter plot
for label in ["subtype_louvain_nuc", "subtype_louvain_cyto"]:
    sc.set_figure_params(figsize = (7, 12))
    ax = sc.pl.scatter(adata_tumor, x = "global_x", y = "global_y", color = label, size = 2, alpha = 1, show = False)
    ax.grid(False)
    # ax.set_xticks([])
    # ax.set_yticks([])
    # ax.set_xlabel("")
    # ax.set_ylabel("")
    # ax.set_title("")
    # for spine in ax.spines.values():
    #     spine.set_visible(False)
    plt.savefig(output_path + f"scatter_{label}.png", dpi = 300, bbox_inches = "tight")
    plt.close()

In [21]:
# DE analysis

